# Final Project — LIMU-BERT Transfer Learning for 3-Class Punch Classification

Fine-tune a pretrained LIMU-BERT encoder (`dapowan/LIMU-BERT-Public`) on our 5-person ~976-file wrist-IMU dataset to classify `hook` / `straight` / `upper`. The notebook mirrors the standalone scripts in `finetuning/` so it can be run end-to-end on Colab.

**Pipeline**:
1. Mount Drive + clone the vendored LIMU-BERT repo
2. Adapter: 100 Hz / 8-channel preprocessing output → 20 Hz / 6-channel / 40-sample LIMU-BERT input
3. Train (frozen mode → partial mode)
4. Evaluate on the standard test split (window-level + file-level + calibration)
5. Leave-one-person-out cross-validation
6. Save a `Predictor` for the game team

Anti-goals (per `instructions.md` §12): no modifying `preprocessing/`, no peeking at the test set for hyperparameter tuning, no shipping a checkpoint without `scaler.pkl` and `config.json`.

## Setup and Configuration

In [ ]:
# Mount Google Drive (skip this cell if running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB = {IN_COLAB}")

In [ ]:
# Install dependencies. PyTorch is preinstalled on Colab.
!pip install -q tsaug seaborn tqdm scikit-learn scipy pandas matplotlib

In [ ]:
# Paths. Sync the project to Drive at this location:
#   {BASE_PATH}/preprocessing/data_cleaning.py
#   {BASE_PATH}/data/{person}/{label}/*.csv
import os, sys
BASE_PATH = "/content/drive/MyDrive/colab/finalproject"
DATA_DIR = f"{BASE_PATH}/data"
ARTIFACTS_DIR = f"{BASE_PATH}/finetuning/artifacts"
CHECKPOINTS_DIR = f"{ARTIFACTS_DIR}/checkpoints"
PLOTS_DIR = f"{ARTIFACTS_DIR}/plots"
METRICS_DIR = f"{ARTIFACTS_DIR}/metrics"
VENDORED_DIR = f"{BASE_PATH}/finetuning/vendored/LIMU-BERT-Public"

for d in (ARTIFACTS_DIR, CHECKPOINTS_DIR, PLOTS_DIR, f"{PLOTS_DIR}/lopo", METRICS_DIR):
    os.makedirs(d, exist_ok=True)

# Make the synced repo importable so we can `from preprocessing.data_cleaning import ...`.
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)

print(f"BASE_PATH    = {BASE_PATH}")
print(f"DATA_DIR     = {DATA_DIR}")
print(f"VENDORED_DIR = {VENDORED_DIR}")
print(f"DATA_DIR exists: {os.path.isdir(DATA_DIR)}")

In [ ]:
# Clone the vendored LIMU-BERT repo if it's not already on Drive.
if not os.path.isdir(VENDORED_DIR):
    os.makedirs(os.path.dirname(VENDORED_DIR), exist_ok=True)
    !git clone --depth 1 https://github.com/dapowan/LIMU-BERT-Public.git "$VENDORED_DIR"
else:
    print(f"LIMU-BERT vendored already present at {VENDORED_DIR}")

# Make the vendored repo importable (its models.py does `from utils import ...`).
if VENDORED_DIR not in sys.path:
    sys.path.insert(0, VENDORED_DIR)

PRETRAIN_CHECKPOINT = f"{VENDORED_DIR}/saved/pretrain_base_hhar_20_120/hhar.pt"
print(f"checkpoint exists: {os.path.isfile(PRETRAIN_CHECKPOINT)}  →  {PRETRAIN_CHECKPOINT}")

In [ ]:
# Common imports
import math, json, pickle, random
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import Literal, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.signal import decimate
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve,
)

# Reuse the preprocessing pipeline unchanged.
from preprocessing.data_cleaning import (
    LABEL_MAP,
    build_manifest,
    load_file,
    lowpass_filter,
    segment,
    train_test_split_by_file,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")

## Configuration Constants

All hyperparameters and adapter decisions live here. Mirrors `finetuning/config.py`.

In [ ]:
# --- Data adapter ---
SOURCE_FS = 100
TARGET_FS = 20
DECIMATION_Q = 5

# Source-side windowing at 100 Hz; after q=5 decimation → 40-sample windows at 20 Hz.
WINDOW_SIZE_100HZ = 200
HOP_SIZE_100HZ = 50

SEQ_LEN = 40          # LIMU-BERT seq length override (was 120 at pretraining)
IN_CHANNELS = 6       # ax, ay, az, gx, gy, gz (drop accel_mag, gyro_mag)

# Pretraining-matching unit conversion.
ACCEL_MG_TO_G = 1.0 / 1000.0
GYRO_DPS_TO_RADPS = math.pi / 180.0
APPLY_ZSCORE_AFTER_UNIT_CONVERT = False   # match pretraining (no z-scoring)

# --- LIMU-BERT model (base_v1 from limu_bert.json) ---
MODEL_HIDDEN = 72
MODEL_HIDDEN_FF = 144
MODEL_N_LAYERS = 4
MODEL_N_HEADS = 4
MODEL_EMB_NORM = True

# --- Classification head ---
HEAD_HIDDEN = 128
HEAD_DROPOUT = 0.3
NUM_CLASSES = 3
LABEL_NAMES = ["hook", "straight", "upper"]

# --- Training ---
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_EPOCHS = 5
VAL_FILE_FRACTION = 0.10

LR_FROZEN = 3e-4
LR_PARTIAL = 5e-5

PEOPLE = ["adnan", "ahmed", "chi", "mercy", "novel"]

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

## Data Adapter — 100 Hz / 8-ch → 20 Hz / 6-ch / 40-sample

`preprocess_file_for_limu_bert(path)` runs `load_file → lowpass(30 Hz) → unit-convert → decimate(q=5)`. The existing `preprocessing/data_cleaning.py` couples normalization and adds magnitude channels we don't want for LIMU-BERT, so we replicate just the load+filter steps and bypass the rest.

LIMU-BERT pretraining (utils.py:254) only divides accel by 9.8 and does **no** gyro normalization and **no** z-scoring. Our `mg / 1000` matches their accel scale; gyro `deg/s × π/180` puts us in rad/s. To match pretraining exactly we skip z-scoring (toggleable).

In [ ]:
def _convert_units(x6_100hz: np.ndarray) -> np.ndarray:
    out = x6_100hz.astype(np.float32, copy=True)
    out[:, :3] *= ACCEL_MG_TO_G          # mg -> g
    out[:, 3:] *= GYRO_DPS_TO_RADPS      # deg/s -> rad/s
    return out


def _decimate_to_20hz(x6_100hz: np.ndarray) -> np.ndarray:
    return decimate(x6_100hz, q=DECIMATION_Q, axis=0, ftype="fir", zero_phase=True).astype(np.float32)


def preprocess_file_for_limu_bert(path) -> np.ndarray:
    raw = load_file(path).to_numpy(dtype=np.float32)
    filtered = lowpass_filter(raw, cutoff_hz=30.0, fs=float(SOURCE_FS), order=4)
    converted = _convert_units(filtered)
    return _decimate_to_20hz(converted)


def _windows_from_manifest(manifest: pd.DataFrame):
    hop_20hz = HOP_SIZE_100HZ // DECIMATION_Q   # 10
    X_chunks, y_chunks, meta_rows = [], [], []
    for _, row in manifest.iterrows():
        x20 = preprocess_file_for_limu_bert(row["path"])
        windows = segment(x20, window_size=SEQ_LEN, hop_size=hop_20hz)
        if windows.shape[0] == 0:
            continue
        label_int = LABEL_MAP[row["label"]]
        X_chunks.append(windows.astype(np.float32, copy=False))
        y_chunks.append(np.full(windows.shape[0], label_int, dtype=np.int64))
        for w in range(windows.shape[0]):
            meta_rows.append({
                "path": str(row["path"]),
                "person": row["person"],
                "label": row["label"],
                "window_idx": w,
            })
    if not X_chunks:
        empty = np.zeros((0, SEQ_LEN, IN_CHANNELS), dtype=np.float32)
        return empty, np.zeros((0,), dtype=np.int64), pd.DataFrame(meta_rows)
    return (
        np.concatenate(X_chunks, axis=0),
        np.concatenate(y_chunks, axis=0),
        pd.DataFrame(meta_rows),
    )


def _fit_scaler_on(X):
    scaler = StandardScaler()
    scaler.fit(X.reshape(-1, X.shape[-1]))
    return scaler


def _apply_scaler(X, scaler):
    flat = scaler.transform(X.reshape(-1, X.shape[-1]))
    return flat.reshape(X.shape).astype(np.float32)


def build_finetune_dataset(data_dir=DATA_DIR, test_ratio=0.2,
                            apply_zscore=APPLY_ZSCORE_AFTER_UNIT_CONVERT,
                            save_scaler_to=None):
    manifest = build_manifest(data_dir)
    train_m, test_m = train_test_split_by_file(manifest, test_ratio=test_ratio, random_state=SEED)
    X_train, y_train, meta_train = _windows_from_manifest(train_m)
    X_test, y_test, meta_test = _windows_from_manifest(test_m)
    scaler = None
    if apply_zscore:
        scaler = _fit_scaler_on(X_train)
        X_train = _apply_scaler(X_train, scaler)
        X_test = _apply_scaler(X_test, scaler)
    if save_scaler_to is not None:
        os.makedirs(os.path.dirname(save_scaler_to), exist_ok=True)
        with open(save_scaler_to, "wb") as f:
            pickle.dump(scaler, f)
    return dict(
        X_train=X_train, y_train=y_train, meta_train=meta_train,
        X_test=X_test, y_test=y_test, meta_test=meta_test, scaler=scaler,
    )


def windows_from_manifest_split(train_m, test_m, apply_zscore=APPLY_ZSCORE_AFTER_UNIT_CONVERT):
    X_train, y_train, meta_train = _windows_from_manifest(train_m)
    X_test, y_test, meta_test = _windows_from_manifest(test_m)
    scaler = None
    if apply_zscore:
        scaler = _fit_scaler_on(X_train)
        X_train = _apply_scaler(X_train, scaler)
        X_test = _apply_scaler(X_test, scaler)
    return dict(
        X_train=X_train, y_train=y_train, meta_train=meta_train,
        X_test=X_test, y_test=y_test, meta_test=meta_test, scaler=scaler,
    )

## Augmentation

Train-time only. 1–2 augmentations per window, each Bernoulli(0.5). The custom 3D rotation applies the *same* matrix to accel and gyro — critical for cross-subject generalization since wrist-strap orientation varies between sessions. No permutation / slicing / mixup — they destroy temporal/class structure for punches.

In [ ]:
from tsaug import Drift, TimeWarp
_drifter = Drift(max_drift=0.2, n_drift_points=5)
_warper  = TimeWarp(n_speed_change=3, max_speed_ratio=1.5)


def _jitter(w, sigma=0.02):
    return w + np.random.normal(0.0, sigma, size=w.shape).astype(w.dtype)


def _scaling(w, sigma=0.1):
    factors = np.random.normal(1.0, sigma, size=(1, w.shape[1])).astype(w.dtype)
    return w * factors


def _drift(w):
    return _drifter.augment(w[np.newaxis, ...])[0].astype(w.dtype)


def _time_warp(w):
    return _warper.augment(w[np.newaxis, ...])[0].astype(w.dtype)


def _random_small_rotation_matrix(max_deg=15.0):
    a = np.deg2rad(np.random.uniform(-max_deg, max_deg, size=3))
    cx, cy, cz = np.cos(a); sx, sy, sz = np.sin(a)
    rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (rz @ ry @ rx).astype(np.float32)


def _rotate(w, max_deg=15.0):
    R = _random_small_rotation_matrix(max_deg)
    out = w.copy()
    out[:, 0:3] = w[:, 0:3] @ R.T
    out[:, 3:6] = w[:, 3:6] @ R.T
    return out


_AUGS = [("jitter", _jitter), ("scaling", _scaling),
         ("drift", _drift), ("time_warp", _time_warp), ("rotation", _rotate)]


def augment_window(window, p_each=0.5, max_n=2):
    indices = np.random.permutation(len(_AUGS))[:max_n]
    out, applied = window, 0
    for idx in indices:
        if np.random.random() < p_each:
            out = _AUGS[idx][1](out)
            applied += 1
    if applied == 0:
        idx = int(np.random.choice(indices))
        out = _AUGS[idx][1](out)
    return out.astype(np.float32, copy=False)

## Model — LIMU-BERT encoder + classification head

Imports the vendored `Transformer` unchanged. Loads the HHAR pretrained checkpoint (largest 6-axis dataset shipped with the repo, 9,166 samples, 9 users) and interpolates the learned positional embeddings from `(120, 72)` → `(40, 72)` via linear interpolation.

**Note on partial mode**: LIMU-BERT applies a single shared attention/FFN block `n_layers=4` times ([models.py:147-155](../vendored/LIMU-BERT-Public/models.py)). There are no per-block parameters, so "unfreeze last 2 blocks" isn't a meaningful operation. `partial` unfreezes the entire shared encoder at the low LR (5e-5); `frozen` keeps every encoder parameter at `requires_grad=False`.

In [ ]:
# Import the vendored Transformer (a 4-layer parameter-shared BERT-style encoder).
from models import Transformer  # noqa: E402


class _LimuCfg:
    def __init__(self, seq_len):
        self.hidden = MODEL_HIDDEN
        self.hidden_ff = MODEL_HIDDEN_FF
        self.feature_num = IN_CHANNELS
        self.n_layers = MODEL_N_LAYERS
        self.n_heads = MODEL_N_HEADS
        self.seq_len = seq_len
        self.emb_norm = MODEL_EMB_NORM


def _interpolate_pos_emb(pretrained_weight, target_seq_len):
    s_old, d = pretrained_weight.shape
    if s_old == target_seq_len:
        return pretrained_weight.clone()
    w = pretrained_weight.t().unsqueeze(0)
    w_new = F.interpolate(w, size=target_seq_len, mode="linear", align_corners=True)
    return w_new.squeeze(0).t().contiguous()


def _load_pretrained_transformer(transformer, checkpoint_path, target_seq_len):
    raw = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    encoder_state = {}
    for k, v in raw.items():
        if k.startswith("transformer."):
            encoder_state[k[len("transformer."):]] = v
    if not encoder_state:
        encoder_state = dict(raw)
    pos_key = "embed.pos_embed.weight"
    if pos_key in encoder_state:
        encoder_state[pos_key] = _interpolate_pos_emb(encoder_state[pos_key], target_seq_len)
    own = transformer.state_dict()
    missing, unexpected = [], []
    for k, v in encoder_state.items():
        if k in own and own[k].shape == v.shape:
            own[k] = v
        else:
            unexpected.append(k)
    for k in own:
        if k not in encoder_state:
            missing.append(k)
    transformer.load_state_dict(own)
    if missing or unexpected:
        print(f"[model] pretrained load — missing: {missing}, unexpected: {unexpected}")


class LimuBertClassifier(nn.Module):
    def __init__(self, mode="frozen"):
        super().__init__()
        self.mode = mode
        cfg = _LimuCfg(seq_len=SEQ_LEN)
        self.encoder = Transformer(cfg)
        self.head_norm = nn.LayerNorm(MODEL_HIDDEN)
        self.head_fc1 = nn.Linear(MODEL_HIDDEN, HEAD_HIDDEN)
        self.head_dropout = nn.Dropout(HEAD_DROPOUT)
        self.head_fc2 = nn.Linear(HEAD_HIDDEN, NUM_CLASSES)
        self._apply_freeze()

    def _apply_freeze(self):
        for p in self.encoder.parameters():
            p.requires_grad = (self.mode == "partial")

    def load_pretrained(self, checkpoint_path=PRETRAIN_CHECKPOINT):
        _load_pretrained_transformer(self.encoder, checkpoint_path, target_seq_len=SEQ_LEN)

    def forward(self, x):
        h = self.encoder(x)
        h = h.mean(dim=1)
        h = self.head_norm(h)
        h = self.head_fc1(h)
        h = F.gelu(h)
        h = self.head_dropout(h)
        return self.head_fc2(h)

    def trainable_params(self):
        return [p for p in self.parameters() if p.requires_grad]


def build_model(mode="frozen", load_pretrained=True, checkpoint_path=None):
    model = LimuBertClassifier(mode=mode)
    if load_pretrained:
        model.load_pretrained(checkpoint_path or PRETRAIN_CHECKPOINT)
        model._apply_freeze()
    return model

## Sanity check — forward pass on dummy data

Verify the model loads pretrained weights, the positional-embedding resize works, and a forward pass produces `(B, 3)` logits.

In [ ]:
_m = build_model("frozen")
dummy = torch.randn(4, SEQ_LEN, IN_CHANNELS)
_out = _m(dummy)
n_train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _m.parameters())
print(f"forward ok: input {tuple(dummy.shape)} -> output {tuple(_out.shape)}")
print(f"trainable params (frozen): {n_train:,} / total {n_total:,}")
del _m, dummy, _out

## Build the fine-tuning dataset

Run the manifest builder + adapter end-to-end. Outputs `(N, 40, 6)` float32 arrays with the file-level 80/20 stratified split from the existing preprocessing pipeline.

In [ ]:
data = build_finetune_dataset(
    data_dir=DATA_DIR,
    save_scaler_to=f"{CHECKPOINTS_DIR}/scaler.pkl",
)
X_all_train = data["X_train"]; y_all_train = data["y_train"]; meta_train = data["meta_train"]
X_test = data["X_test"]; y_test = data["y_test"]; meta_test = data["meta_test"]

print(f"X_train: {X_all_train.shape}  y_train: {y_all_train.shape}")
print(f"X_test : {X_test.shape}  y_test : {y_test.shape}")
print("per-channel train mean:", X_all_train.reshape(-1, X_all_train.shape[-1]).mean(axis=0))
print("per-channel train std :", X_all_train.reshape(-1, X_all_train.shape[-1]).std(axis=0))
print("label counts (train):", dict(zip(*np.unique(y_all_train, return_counts=True))))
print("label counts (test ):", dict(zip(*np.unique(y_test, return_counts=True))))

## Training loop (frozen or partial)

Carves a 10% validation set from the training files (stratified, file-level — never window-level) for early stopping. AdamW + 5-epoch warmup + cosine schedule. Mixed precision on GPU. Class weights are inverse-frequency from `y_train`.

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X; self.y = y; self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            x = augment_window(x)
        return torch.from_numpy(x.astype(np.float32, copy=False)), int(self.y[idx])


def _split_train_val_by_file(meta_train, val_fraction, seed):
    file_df = meta_train.groupby("path").agg(label=("label", "first")).reset_index()
    train_files, val_files = train_test_split(
        file_df["path"].to_numpy(),
        test_size=val_fraction,
        stratify=file_df["label"].to_numpy(),
        random_state=seed,
    )
    val_mask = meta_train["path"].isin(set(val_files.tolist())).to_numpy()
    return ~val_mask, val_mask


def inverse_freq_weights(y, num_classes=NUM_CLASSES):
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts = np.where(counts == 0, 1.0, counts)
    inv = 1.0 / counts
    inv = inv * (num_classes / inv.sum())
    return torch.tensor(inv, dtype=torch.float32)


def cosine_warmup_lr(epoch, warmup_epochs, max_epochs):
    if epoch < warmup_epochs:
        return (epoch + 1) / max(1, warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


@dataclass
class TrainResult:
    best_val_f1: float
    history: list
    checkpoint_path: str


def train_one_run(X_train, y_train, X_val, y_val, mode, max_epochs, batch_size, lr,
                  save_to, patience=EARLY_STOP_PATIENCE, use_augment=True, verbose=True):
    print(f"[train] device={DEVICE}, mode={mode}, lr={lr}, epochs<= {max_epochs}")
    model = build_model(mode=mode, load_pretrained=True).to(DEVICE)
    train_loader = DataLoader(WindowDataset(X_train, y_train, augment=use_augment),
                              batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(WindowDataset(X_val, y_val, augment=False),
                            batch_size=batch_size, shuffle=False, num_workers=0)
    class_weights = inverse_freq_weights(y_train).to(DEVICE)
    print(f"[train] class weights: {class_weights.cpu().numpy()}")
    optimizer = torch.optim.AdamW(model.trainable_params(), lr=lr, weight_decay=WEIGHT_DECAY)
    use_amp = DEVICE.type == "cuda"
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_f1, best_epoch, stale = -1.0, -1, 0
    history = []
    for epoch in range(max_epochs):
        lr_scale = cosine_warmup_lr(epoch, WARMUP_EPOCHS, max_epochs)
        for g in optimizer.param_groups:
            g["lr"] = lr * lr_scale

        model.train()
        train_loss, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb, weight=class_weights)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.trainable_params(), GRAD_CLIP)
            scaler_amp.step(optimizer); scaler_amp.update()
            train_loss += loss.item() * xb.size(0); n += xb.size(0)
        train_loss /= max(1, n)

        model.eval()
        val_loss = 0.0; all_p, all_l = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_loss += F.cross_entropy(logits, yb, weight=class_weights).item() * xb.size(0)
                all_p.append(logits.argmax(-1).cpu().numpy())
                all_l.append(yb.cpu().numpy())
        val_loss /= max(1, len(val_loader.dataset))
        y_pred = np.concatenate(all_p) if all_p else np.zeros(0, dtype=np.int64)
        y_true = np.concatenate(all_l) if all_l else np.zeros(0, dtype=np.int64)
        val_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

        history.append(dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"],
                            train_loss=train_loss, val_loss=val_loss, val_macro_f1=val_f1))
        if verbose:
            print(f"[epoch {epoch:3d}] lr={optimizer.param_groups[0]['lr']:.2e}  "
                  f"train={train_loss:.4f}  val={val_loss:.4f}  valF1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1, best_epoch, stale = val_f1, epoch, 0
            os.makedirs(os.path.dirname(save_to), exist_ok=True)
            torch.save({"model_state": model.state_dict(), "mode": mode, "epoch": epoch,
                        "val_macro_f1": val_f1, "seq_len": SEQ_LEN, "in_channels": IN_CHANNELS},
                       save_to)
        else:
            stale += 1
            if stale >= patience:
                print(f"[train] early stop at {epoch} (best={best_epoch}, f1={best_f1:.4f})")
                break

    return TrainResult(best_val_f1=best_f1, history=history, checkpoint_path=save_to)


def plot_training_curves(history, out_path):
    epochs = [h["epoch"] for h in history]
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axs[0].plot(epochs, [h["val_loss"]  for h in history], label="val")
    axs[0].set_xlabel("epoch"); axs[0].set_ylabel("loss"); axs[0].legend(); axs[0].set_title("loss")
    axs[1].plot(epochs, [h["val_macro_f1"] for h in history])
    axs[1].set_xlabel("epoch"); axs[1].set_ylabel("val macro-F1"); axs[1].set_title("val macro-F1")
    fig.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, dpi=120); plt.close(fig)
    return out_path

## Train — Frozen mode

Encoder weights stay locked; only the classification head trains. This is the cleaner transfer-learning signal and the harder bar.

In [ ]:
train_mask, val_mask = _split_train_val_by_file(meta_train, VAL_FILE_FRACTION, SEED)
X_train, y_train = X_all_train[train_mask], y_all_train[train_mask]
X_val,   y_val   = X_all_train[val_mask],   y_all_train[val_mask]
print(f"train windows={len(y_train)}  val windows={len(y_val)}  test windows={len(y_test)}")

ckpt_frozen = f"{CHECKPOINTS_DIR}/limu_bert_frozen.pt"
result_frozen = train_one_run(
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    mode="frozen", max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    lr=LR_FROZEN, save_to=ckpt_frozen,
)
plot_training_curves(result_frozen.history, f"{PLOTS_DIR}/training_curves_frozen.png")
print(f"[done] best val macro-F1: {result_frozen.best_val_f1:.4f}  ckpt: {ckpt_frozen}")

## Evaluate — Frozen mode

Window-level metrics (acc, macro/weighted F1, per-class P/R/F1, confusion, OvR ROC/AUC), file-level majority-vote metrics (closer to what the game sees), and a confidence threshold tuned to keep the false-trigger rate <5% on incorrect predictions.

In [ ]:
@torch.no_grad()
def predict_probs(model, X, batch_size=256):
    model.eval().to(DEVICE)
    out = []
    for s in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[s:s+batch_size].astype(np.float32)).to(DEVICE)
        out.append(F.softmax(model(xb), dim=-1).cpu().numpy())
    return np.concatenate(out, axis=0) if out else np.zeros((0, NUM_CLASSES), dtype=np.float32)


def save_confusion(cm, labels, out_png, title):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    fig.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=120); plt.close(fig)


def save_roc(probs, y_true, labels, out_png):
    y_bin = label_binarize(y_true, classes=list(range(len(labels))))
    aucs = {}
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, name in enumerate(labels):
        if y_bin[:, i].sum() == 0:
            aucs[name] = float("nan"); continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        auc = roc_auc_score(y_bin[:, i], probs[:, i])
        aucs[name] = float(auc)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("one-vs-rest ROC")
    ax.legend(loc="lower right")
    fig.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=120); plt.close(fig)
    return aucs


def file_level_majority_vote(probs, y_true, meta):
    preds = probs.argmax(axis=-1)
    df = meta.copy(); df["pred"] = preds; df["true"] = y_true
    fp, ft = [], []
    for _, g in df.groupby("path"):
        fp.append(Counter(g["pred"].tolist()).most_common(1)[0][0])
        ft.append(g["true"].iloc[0])
    return np.array(ft), np.array(fp)


def confidence_threshold(probs, y_true, target_fp_rate=0.05):
    preds = probs.argmax(axis=-1); confs = probs.max(axis=-1)
    incorrect = preds != y_true
    if incorrect.sum() == 0:
        return 0.5
    inc_confs = confs[incorrect]
    grid = np.linspace(1.0 / NUM_CLASSES, 1.0, 100)
    best_t, best_rate = float(grid[-1]), 1.0
    for t in grid:
        rate = float((inc_confs >= t).mean())
        if rate < target_fp_rate:
            return float(t)
        if rate < best_rate:
            best_rate, best_t = rate, float(t)
    return best_t


def calibration_plot(probs, y_true, threshold, out_png):
    preds = probs.argmax(axis=-1); confs = probs.max(axis=-1)
    correct = preds == y_true
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(confs[correct], bins=30, alpha=0.6, label="correct", color="tab:green")
    ax.hist(confs[~correct], bins=30, alpha=0.6, label="incorrect", color="tab:red")
    ax.axvline(threshold, color="black", linestyle="--", label=f"threshold={threshold:.3f}")
    ax.set_xlabel("softmax max"); ax.set_ylabel("count")
    ax.set_title("confidence histogram"); ax.legend()
    fig.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=120); plt.close(fig)


def evaluate_mode(mode):
    ckpt_path = f"{CHECKPOINTS_DIR}/limu_bert_{mode}.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_model(mode=mode, load_pretrained=False)
    model.load_state_dict(ckpt["model_state"])
    probs = predict_probs(model, X_test)
    preds = probs.argmax(axis=-1)

    p, r, f, _ = precision_recall_fscore_support(
        y_test, preds, labels=list(range(NUM_CLASSES)), zero_division=0)
    win = {
        "accuracy": float((preds == y_test).mean()),
        "macro_f1": float(f1_score(y_test, preds, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_test, preds, average="weighted", zero_division=0)),
        "per_class": {LABEL_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]),
                                       "f1": float(f[i])} for i in range(NUM_CLASSES)},
    }
    cm_w = confusion_matrix(y_test, preds, labels=list(range(NUM_CLASSES)))
    save_confusion(cm_w, LABEL_NAMES, f"{PLOTS_DIR}/confusion_window_{mode}.png",
                   f"window-level ({mode})")
    win["confusion_matrix"] = cm_w.tolist()
    win["roc_auc"] = save_roc(probs, y_test, LABEL_NAMES, f"{PLOTS_DIR}/roc_{mode}.png")

    y_ft, y_fp = file_level_majority_vote(probs, y_test, meta_test)
    file_metrics = {
        "accuracy": float((y_fp == y_ft).mean()),
        "macro_f1": float(f1_score(y_ft, y_fp, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_ft, y_fp, average="weighted", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_ft, y_fp, labels=list(range(NUM_CLASSES))).tolist(),
    }
    cm_f = confusion_matrix(y_ft, y_fp, labels=list(range(NUM_CLASSES)))
    save_confusion(cm_f, LABEL_NAMES, f"{PLOTS_DIR}/confusion_file_{mode}.png",
                   f"file-level ({mode})")

    t = confidence_threshold(probs, y_test, target_fp_rate=0.05)
    calibration_plot(probs, y_test, t, f"{PLOTS_DIR}/confidence_hist_{mode}.png")

    metrics = {
        "mode": mode,
        "window_level": win,
        "file_level": file_metrics,
        "confidence_threshold_5pct_ft": float(t),
        "n_test_windows": int(len(y_test)),
        "n_test_files": int(meta_test["path"].nunique()),
    }
    os.makedirs(METRICS_DIR, exist_ok=True)
    with open(f"{METRICS_DIR}/{mode}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"[eval] {mode}  window F1={win['macro_f1']:.4f}  "
          f"file F1={file_metrics['macro_f1']:.4f}  threshold={t:.3f}")
    return metrics, probs


frozen_metrics, frozen_probs = evaluate_mode("frozen")

### Display saved plots — frozen

In [ ]:
from IPython.display import Image, display
for name in ("training_curves_frozen", "confusion_window_frozen",
             "confusion_file_frozen", "roc_frozen", "confidence_hist_frozen"):
    p = f"{PLOTS_DIR}/{name}.png"
    if os.path.exists(p):
        print(name)
        display(Image(filename=p))

## Train — Partial mode

Unfreezes the (parameter-shared) encoder at the low LR (5e-5). Usually wins on absolute accuracy but is closer to "training from scratch" since LIMU-BERT uses parameter sharing across layers.

In [ ]:
ckpt_partial = f"{CHECKPOINTS_DIR}/limu_bert_partial.pt"
result_partial = train_one_run(
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    mode="partial", max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    lr=LR_PARTIAL, save_to=ckpt_partial,
)
plot_training_curves(result_partial.history, f"{PLOTS_DIR}/training_curves_partial.png")
print(f"[done] best val macro-F1: {result_partial.best_val_f1:.4f}  ckpt: {ckpt_partial}")

## Evaluate — Partial mode

In [ ]:
partial_metrics, partial_probs = evaluate_mode("partial")

### Display saved plots — partial

In [ ]:
for name in ("training_curves_partial", "confusion_window_partial",
             "confusion_file_partial", "roc_partial", "confidence_hist_partial"):
    p = f"{PLOTS_DIR}/{name}.png"
    if os.path.exists(p):
        print(name)
        display(Image(filename=p))

## Save hyperparameter snapshot

`config.json` next to the checkpoints — the game team needs this to reproduce preprocessing from disk alone.

In [ ]:
config_snapshot = {
    "lr_frozen": LR_FROZEN,
    "lr_partial": LR_PARTIAL,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "grad_clip": GRAD_CLIP,
    "seq_len": SEQ_LEN,
    "in_channels": IN_CHANNELS,
    "source_fs": SOURCE_FS,
    "target_fs": TARGET_FS,
    "decimation_q": DECIMATION_Q,
    "window_size_100hz": WINDOW_SIZE_100HZ,
    "hop_size_100hz": HOP_SIZE_100HZ,
    "apply_zscore_after_unit_convert": APPLY_ZSCORE_AFTER_UNIT_CONVERT,
    "pretrain_checkpoint": str(PRETRAIN_CHECKPOINT),
    "label_names": LABEL_NAMES,
    "seed": SEED,
}
with open(f"{CHECKPOINTS_DIR}/config.json", "w") as f:
    json.dump(config_snapshot, f, indent=2)
print(f"wrote {CHECKPOINTS_DIR}/config.json")

## Leave-One-Person-Out Cross-Validation

The 80/20 file split mixes all 5 people in both train and test, so good numbers there can still hide person-specific overfitting. The game has to work on a 6th player it has never seen. For each held-out person, train fresh (frozen mode only — partial is too expensive for LOPO) on the other 4 and test on this one. If LOPO macro-F1 is dramatically below the standard-split number, that's the headline finding.

Expect ~30 min – 2 hr depending on hardware. Reduce `LOPO_MAX_EPOCHS` for a faster run.

In [ ]:
LOPO_MAX_EPOCHS = MAX_EPOCHS   # set lower for a faster sweep

manifest_all = build_manifest(DATA_DIR)
rows = []
for person in PEOPLE:
    print(f"\n========== LOPO held-out = {person} ==========")
    train_m = manifest_all[manifest_all["person"] != person].reset_index(drop=True)
    test_m  = manifest_all[manifest_all["person"] == person].reset_index(drop=True)
    if len(test_m) == 0:
        print(f"[lopo] no files for {person}, skipping"); continue

    d = windows_from_manifest_split(train_m, test_m)
    X_all, y_all, m_all = d["X_train"], d["y_train"], d["meta_train"]
    X_tst, y_tst, m_tst = d["X_test"],  d["y_test"],  d["meta_test"]
    tm, vm = _split_train_val_by_file(m_all, VAL_FILE_FRACTION, SEED)
    Xtr, ytr = X_all[tm], y_all[tm]
    Xvl, yvl = X_all[vm], y_all[vm]
    print(f"[lopo] train={len(ytr)}  val={len(yvl)}  test={len(y_tst)}  "
          f"test files={m_tst['path'].nunique()}")

    ckpt_path = f"{CHECKPOINTS_DIR}/lopo_frozen_{person}.pt"
    res = train_one_run(Xtr, ytr, Xvl, yvl, mode="frozen",
                        max_epochs=LOPO_MAX_EPOCHS, batch_size=BATCH_SIZE,
                        lr=LR_FROZEN, save_to=ckpt_path, verbose=False)

    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_model(mode="frozen", load_pretrained=False)
    model.load_state_dict(ck["model_state"])
    probs = predict_probs(model, X_tst)
    if len(probs) == 0:
        print(f"[lopo] no test windows for {person}"); continue
    preds = probs.argmax(axis=-1)
    win_f1 = float(f1_score(y_tst, preds, average="macro", zero_division=0))
    y_ft, y_fp = file_level_majority_vote(probs, y_tst, m_tst)
    file_f1 = float(f1_score(y_ft, y_fp, average="macro", zero_division=0))

    cm = confusion_matrix(y_tst, preds, labels=list(range(NUM_CLASSES)))
    save_confusion(cm, LABEL_NAMES, f"{PLOTS_DIR}/lopo/{person}_confusion.png",
                   f"LOPO held-out={person}")
    rows.append({
        "person": person,
        "n_test_windows": int(len(y_tst)),
        "n_test_files": int(m_tst["path"].nunique()),
        "val_macro_f1": float(res.best_val_f1),
        "test_window_macro_f1": win_f1,
        "test_file_macro_f1": file_f1,
    })
    print(f"[lopo] {person}: win-F1={win_f1:.4f}  file-F1={file_f1:.4f}  "
          f"val-F1={res.best_val_f1:.4f}")

df_lopo = pd.DataFrame(rows)
df_lopo.to_csv(f"{METRICS_DIR}/lopo_results.csv", index=False)
summary = {
    "window_macro_f1_mean": float(df_lopo["test_window_macro_f1"].mean()) if len(df_lopo) else None,
    "window_macro_f1_std":  float(df_lopo["test_window_macro_f1"].std(ddof=0)) if len(df_lopo) else None,
    "file_macro_f1_mean":   float(df_lopo["test_file_macro_f1"].mean()) if len(df_lopo) else None,
    "file_macro_f1_std":    float(df_lopo["test_file_macro_f1"].std(ddof=0)) if len(df_lopo) else None,
    "per_person": rows,
}
with open(f"{METRICS_DIR}/lopo_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
df_lopo

### LOPO summary

In [ ]:
if len(df_lopo):
    print(f"window macroF1 mean ± std: {summary['window_macro_f1_mean']:.4f} ± {summary['window_macro_f1_std']:.4f}")
    print(f"file   macroF1 mean ± std: {summary['file_macro_f1_mean']:.4f} ± {summary['file_macro_f1_std']:.4f}")
for p in PEOPLE:
    path = f"{PLOTS_DIR}/lopo/{p}_confusion.png"
    if os.path.exists(path):
        print(p); display(Image(filename=path))

## Inference helper for the game team

Side-effect-free at import time: no model loads on `import`. Use `Predictor.from_disk(...)` to instantiate.

**Contract**: this model was trained to match LIMU-BERT pretraining inputs (accel in g, gyro in rad/s, decimated to 20 Hz, no z-scoring). The game team must pass **raw-units** windows at 100 Hz (mg, deg/s) — NOT the output of `preprocessing.preprocess_file()` (which has been z-scored).

Recommended live-inference pipeline:
```python
from preprocessing.data_cleaning import load_file, lowpass_filter, segment
raw = load_file(csv_path).to_numpy(np.float32)
filtered = lowpass_filter(raw, cutoff_hz=30.0, fs=100.0)
windows = segment(filtered, window_size=200, hop_size=50)
predictor = Predictor.from_disk()
label, conf = predictor.predict_window(windows[i])
```

In [ ]:
@dataclass
class Predictor:
    model: nn.Module
    device: torch.device
    threshold: float = 0.0
    label_names: tuple = tuple(LABEL_NAMES)

    @classmethod
    def from_disk(cls, mode="frozen", checkpoint_path=None, threshold=None):
        device = DEVICE
        ckpt_path = checkpoint_path or f"{CHECKPOINTS_DIR}/limu_bert_{mode}.pt"
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = build_model(mode=ckpt.get("mode", mode), load_pretrained=False)
        model.load_state_dict(ckpt["model_state"])
        model.eval().to(device)
        if threshold is None:
            mpath = f"{METRICS_DIR}/{mode}_metrics.json"
            threshold = 0.0
            if os.path.exists(mpath):
                with open(mpath) as f:
                    threshold = float(json.load(f).get("confidence_threshold_5pct_ft", 0.0))
        return cls(model=model, device=device, threshold=threshold)

    def _to_lb_input(self, window_100hz_6ch):
        arr = window_100hz_6ch.astype(np.float32, copy=True)
        if arr.ndim != 2 or arr.shape[1] != 6:
            raise ValueError(f"expected (T, 6) at 100 Hz; got {window_100hz_6ch.shape}")
        converted = _convert_units(arr)
        decimated = _decimate_to_20hz(converted)
        if decimated.shape[0] < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - decimated.shape[0], IN_CHANNELS), dtype=np.float32)
            decimated = np.concatenate([decimated, pad], axis=0)
        decimated = decimated[:SEQ_LEN]
        return torch.from_numpy(decimated).unsqueeze(0).to(self.device)

    @torch.no_grad()
    def predict_window(self, window_100hz_6ch):
        x = self._to_lb_input(window_100hz_6ch)
        logits = self.model(x)
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
        idx = int(probs.argmax()); conf = float(probs[idx])
        if conf < self.threshold:
            return ("none", conf)
        return (self.label_names[idx], conf)


def predict_punch(window_100hz_8ch, predictor):
    if window_100hz_8ch.ndim != 2 or window_100hz_8ch.shape[1] not in (6, 8):
        raise ValueError(f"expected (T, 6) or (T, 8); got {window_100hz_8ch.shape}")
    return predictor.predict_window(window_100hz_8ch[:, :6])

### Inference demo

In [ ]:
predictor = Predictor.from_disk(mode="frozen")
print(f"loaded frozen model — confidence threshold = {predictor.threshold:.3f}")

# Demo on first window of first test file.
first_path = meta_test["path"].iloc[0]
raw = load_file(first_path).to_numpy(np.float32)
filtered = lowpass_filter(raw, cutoff_hz=30.0, fs=100.0)
windows = segment(filtered, window_size=WINDOW_SIZE_100HZ, hop_size=HOP_SIZE_100HZ)
print(f"file: {first_path}")
print(f"true label: {meta_test['label'].iloc[0]}")
print(f"windows in file: {windows.shape}")
for i in range(min(3, len(windows))):
    label, conf = predictor.predict_window(windows[i])
    print(f"  window {i}: pred={label}  conf={conf:.3f}")

## Final results summary

Per `instructions.md` §12: **never report a standard-split number without LOPO next to it.** A 95% standard-split with 60% LOPO is *worse* than 85% / 80%, not better.

In [ ]:
summary_table = pd.DataFrame([
    {
        "mode": "frozen",
        "window_macro_f1": frozen_metrics["window_level"]["macro_f1"],
        "file_macro_f1":   frozen_metrics["file_level"]["macro_f1"],
        "confidence_thresh_5pct_ft": frozen_metrics["confidence_threshold_5pct_ft"],
        "lopo_window_f1_mean_std": (
            f"{summary['window_macro_f1_mean']:.4f} ± {summary['window_macro_f1_std']:.4f}"
            if summary["window_macro_f1_mean"] is not None else "—"
        ),
        "lopo_file_f1_mean_std": (
            f"{summary['file_macro_f1_mean']:.4f} ± {summary['file_macro_f1_std']:.4f}"
            if summary["file_macro_f1_mean"] is not None else "—"
        ),
    },
    {
        "mode": "partial",
        "window_macro_f1": partial_metrics["window_level"]["macro_f1"],
        "file_macro_f1":   partial_metrics["file_level"]["macro_f1"],
        "confidence_thresh_5pct_ft": partial_metrics["confidence_threshold_5pct_ft"],
        "lopo_window_f1_mean_std": "— (skipped per spec)",
        "lopo_file_f1_mean_std":   "— (skipped per spec)",
    },
])
summary_table

In [ ]:
import shutil, zipfile
EXPORT_DIR = f"{BASE_PATH}/export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# copy the vendored LIMU-BERT source the predictor needs at import time
shutil.copytree(f"{VENDORED_DIR}", f"{EXPORT_DIR}/LIMU-BERT-Public", dirs_exist_ok=True)
shutil.copy(f"{CHECKPOINTS_DIR}/limu_bert_frozen.pt", EXPORT_DIR)
shutil.copy(f"{CHECKPOINTS_DIR}/config.json", EXPORT_DIR)
shutil.copy(f"{CHECKPOINTS_DIR}/scaler.pkl", EXPORT_DIR)
shutil.copy(f"{METRICS_DIR}/frozen_metrics.json", EXPORT_DIR)  # for the threshold

with zipfile.ZipFile(f"{BASE_PATH}/demo_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(EXPORT_DIR):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, EXPORT_DIR))
print("done →", f"{BASE_PATH}/demo_bundle.zip")